# Evaluation Notebook

This file performs evaluation of all algorithms/variants, stores results in a Zarr file, and generates plots.

## Configuration

In [1]:
# Set the output file automatically.
from datetime import datetime
datetime_str = datetime.now().strftime("%Y%m%d_%H%M%S")
output_file = "./results/patrolling_comms_" + datetime_str + ".zarr"

# output_file = "/data/anthony/dev/qmas_task_allocation/scripts/final/results/patrolling_comms_20260428_014357.zarr"
# output_file = "/data/anthony/dev/qmas_task_allocation/scripts/final/results/patrolling_comms_20260428_111309.zarr"
# output_file = '/home/anthony/data/dev/qmas_task_allocation/scripts/final/results/patrolling_comms_20260428_133357.zarr'
# output_file = '/home/anthony/data/dev/qmas_task_allocation/scripts/final/results/patrolling_comms_20260428_165714.zarr'

# output_file = '/home/anthony/data/dev/qmas_task_allocation/scripts/final/results/patrolling_comms_20260428_231726.zarr

output_file = "./results/patrolling_comms_test8_pareto.zarr"

# Pareto evaluation: sweep comms_probability for baselines, single point for learned comms.
pareto_output_file = "./results/patrolling_comms_test8_pareto_curve.zarr"
pareto_obs_radii = [50.0, 100.0, 150.0, 200.0]
comms_probabilities_sweep = [0.005, 0.01, 0.02, 0.043, 0.08, 0.15, 0.3, 0.5]


## Setup

In [2]:
%reload_ext autoreload
%autoreload 2
from onpolicy.scripts.eval.eval_pettingzoo import main
import os
import time
from multiprocessing import Process
from args import *

MAX_PROCESSES = 25

# Set default arguments for evaluation. Feel free to change these!
default_args = {}
default_args["eval_output_file"] = wrap_arg_val(output_file)
default_args["cuda"] = wrap_arg_val(True)
default_args["cuda_idx"] = wrap_arg_val(2)
default_args["cuda_idx_predictor"] = default_args["cuda_idx"]
default_args["eval_episodes"] = wrap_arg_val(3)
default_args["episode_length"] = wrap_arg_val(200)
default_args["seed"] = wrap_arg_val(2)

default_args["observation_mask"] = wrap_arg_val(True)

default_args["diffusion_autoregression_steps"] = wrap_arg_val(1)

# default_args["graph_name"] = wrap_arg_val("cumberland")
# default_args["graph_file"] = wrap_arg_val("../../patrolling_zoo/graphs/cumberland.graph")
default_args["graph_random"] = wrap_arg_val(True)

default_args["observation_radius_random_min"] = wrap_arg_val(0.0)
default_args["observation_radius_random_max"] = wrap_arg_val(0.0)

# Evaluation

In [ ]:
# obs_radii = [0.0, 50.0, 100.0, 150.0, 200.0, 250.0, 300.0]
obs_radii = [50.0, 100.0, 150.0, 200.0]
comms_probability = 0.043

import numpy as np

# Check if the output file already exists. If so, skip this cell.
if os.path.exists(output_file):
    print("Output file already exists. Skipping evaluation.")

else:
    processes = []

    def start_process(args):
        p = Process(target=main, args=[convert_args(args)])
        processes.append(p)

    for obs_radius in obs_radii:
        args = load_args("/data/group/mas/qmas/results/patrolling_zoo_v0/rmappo/patrolling-randomOR-noSkip-draft1/wandb/run-20260429_003713-uozeh7d7/files/episode_39", default_args)
        args["eval_series"] = "MAGEC"
        args["experiment_name"] = wrap_arg_val(f"obs_radius_{obs_radius:.1f}")
        args["observation_radius"] = wrap_arg_val(obs_radius)
        args["communication_probability"] = wrap_arg_val(comms_probability)
        # args["seed"] = wrap_arg_val(np.random.randint(0, 10000))
        start_process(args)

        args = load_args("/data/group/mas/qmas/results/patrolling_zoo_v0/rmappo/patrolling-randomOR-noSkip-pred-draft1/wandb/run-20260429_011457-4vd6rl3j/files/episode_48", default_args)
        args["eval_series"] = "Prediction"
        args["experiment_name"] = wrap_arg_val(f"obs_radius_{obs_radius:.1f}")
        args["observation_radius"] = wrap_arg_val(obs_radius)
        args["communication_probability"] = wrap_arg_val(comms_probability)
        # args["seed"] = wrap_arg_val(np.random.randint(0, 10000))
        start_process(args)

        # args = load_args("/data/group/mas/qmas/results/patrolling_zoo_v0/rmappo/patrolling-randomOR-pred-uqstd-draft1/wandb/run-20260428_182546-v688m1pp/files", default_args)
        # args["eval_series"] = "Prediction + UQ"
        # args["experiment_name"] = wrap_arg_val(f"obs_radius_{obs_radius:.1f}")
        # args["observation_radius"] = wrap_arg_val(obs_radius)
        # args["communication_probability"] = wrap_arg_val(comms_probability)
        # # args["seed"] = wrap_arg_val(np.random.randint(0, 10000))
        # start_process(args)

        args = load_args("/data/group/mas/qmas/results/patrolling_zoo_v0/rmappo/patrolling-randomOR-noSkip-comms-pred-draft1/wandb/run-20260429_011412-55vkqb6a/files/episode_48", default_args)
        args["eval_series"] = "Learned Comms. + Prediction"
        args["experiment_name"] = wrap_arg_val(f"obs_radius_{obs_radius:.1f}")
        args["observation_radius"] = wrap_arg_val(obs_radius)
        args["communication_probability"] = wrap_arg_val(comms_probability)
        # args["seed"] = wrap_arg_val(np.random.randint(0, 10000))
        start_process(args)

        args = load_args("/data/group/mas/qmas/results/patrolling_zoo_v0/rmappo/patrolling-randomOR-noSkip-comms-pred-uqstd-draft1/wandb/run-20260429_003704-47915ufa/files/episode_55", default_args)
        args["eval_series"] = f"Learned Comms. + Prediction + UQ"
        args["experiment_name"] = wrap_arg_val(f"obs_radius_{obs_radius:.1f}")
        args["observation_radius"] = wrap_arg_val(obs_radius)
        args["communication_probability"] = wrap_arg_val(comms_probability)
        # args["seed"] = wrap_arg_val(np.random.randint(0, 10000))
        start_process(args)

    # JOIN ALL PROCESSES
    # Up to MAX_PROCESSES processes can run simultaneously.
    running_processes = []
    time_start = time.time()
    failed_processes = []
    
    while len(processes) > 0 or len(running_processes) > 0:
        # Start new processes if we have capacity.
        while len(running_processes) < MAX_PROCESSES and len(processes) > 0:
            p = processes.pop(0)
            p.start()
            running_processes.append(p)
        
        # Check for finished processes.
        for p in running_processes:
            if not p.is_alive():
                p.join()
                if p.exitcode != 0:
                    failed_processes.append(p)
        running_processes = [p for p in running_processes if p.is_alive()]
    
    time_end = time.time()
    print(f"Total evaluation time: {time_end - time_start:.2f} seconds")
    
    if failed_processes:
        raise RuntimeError(f"{len(failed_processes)} subprocess(es) failed with non-zero exit codes")
    
    print("All processes finished successfully!")

## Pareto Evaluation

Sweeps `comms_probability` for random-comms baselines (MAGEC, Prediction) across all observation radii.
Learned comms runs are evaluated at their natural operating point (single point per obs_radius).
Results saved to `pareto_output_file` separately from the main evaluation.

In [3]:
import numpy as np

# Check if the pareto output file already exists. If so, skip this cell.
if os.path.exists(pareto_output_file):
    print("Pareto output file already exists. Skipping evaluation.")

else:
    pareto_processes = []

    def start_pareto_process(args):
        p = Process(target=main, args=[convert_args(args)])
        pareto_processes.append(p)

    for obs_radius in pareto_obs_radii:

        # --- Sweep comms_probability for random-comms baselines ---
        for comms_p in comms_probabilities_sweep:
            args = load_args("/data/group/mas/qmas/results/patrolling_zoo_v0/rmappo/patrolling-randomOR-noSkip-draft1/wandb/run-20260429_003713-uozeh7d7/files/episode_39", default_args)
            args["eval_output_file"] = wrap_arg_val(pareto_output_file)
            args["eval_series"] = "MAGEC"
            args["experiment_name"] = wrap_arg_val(f"obs_radius_{obs_radius:.1f}_comms_{comms_p:.3f}")
            args["observation_radius"] = wrap_arg_val(obs_radius)
            args["communication_probability"] = wrap_arg_val(comms_p)
            start_pareto_process(args)

            args = load_args("/data/group/mas/qmas/results/patrolling_zoo_v0/rmappo/patrolling-randomOR-noSkip-pred-draft1/wandb/run-20260429_011457-4vd6rl3j/files/episode_48", default_args)
            args["eval_output_file"] = wrap_arg_val(pareto_output_file)
            args["eval_series"] = "Prediction"
            args["experiment_name"] = wrap_arg_val(f"obs_radius_{obs_radius:.1f}_comms_{comms_p:.3f}")
            args["observation_radius"] = wrap_arg_val(obs_radius)
            args["communication_probability"] = wrap_arg_val(comms_p)
            start_pareto_process(args)

        # --- Learned comms: single operating point per obs_radius ---
        args = load_args("/data/group/mas/qmas/results/patrolling_zoo_v0/rmappo/patrolling-randomOR-noSkip-comms-pred-draft1/wandb/run-20260429_011412-55vkqb6a/files/episode_48", default_args)
        args["eval_output_file"] = wrap_arg_val(pareto_output_file)
        args["eval_series"] = "Learned Comms. + Prediction"
        args["experiment_name"] = wrap_arg_val(f"obs_radius_{obs_radius:.1f}")
        args["observation_radius"] = wrap_arg_val(obs_radius)
        start_pareto_process(args)

        args = load_args("/data/group/mas/qmas/results/patrolling_zoo_v0/rmappo/patrolling-randomOR-noSkip-comms-pred-uqstd-draft1/wandb/run-20260429_003704-47915ufa/files/episode_55", default_args)
        args["eval_output_file"] = wrap_arg_val(pareto_output_file)
        args["eval_series"] = "Learned Comms. + Prediction + UQ"
        args["experiment_name"] = wrap_arg_val(f"obs_radius_{obs_radius:.1f}")
        args["observation_radius"] = wrap_arg_val(obs_radius)
        start_pareto_process(args)

    # JOIN ALL PROCESSES
    running_processes = []
    time_start = time.time()
    failed_processes = []

    while len(pareto_processes) > 0 or len(running_processes) > 0:
        while len(running_processes) < MAX_PROCESSES and len(pareto_processes) > 0:
            p = pareto_processes.pop(0)
            p.start()
            running_processes.append(p)

        for p in running_processes:
            if not p.is_alive():
                p.join()
                if p.exitcode != 0:
                    failed_processes.append(p)
        running_processes = [p for p in running_processes if p.is_alive()]

    time_end = time.time()
    print(f"Total Pareto evaluation time: {time_end - time_start:.2f} seconds")

    if failed_processes:
        raise RuntimeError(f"{len(failed_processes)} subprocess(es) failed with non-zero exit codes")

    print("All Pareto processes finished successfully!")


Converted arguments: ['--accept_ratio', '0.5', '--action_full_max_nodes', '40', '--action_method', 'neighbors', '--action_neighbors_max_degree', '10', '--agent_speed', '40.0', '--algorithm_class', 'qmas.variant_two_modules.new.algorithm.QmasAlgorithm', '--algorithm_name', 'rmappo', '--alpha', '1.0', '--attrition_fixed_times', '[]', '--attrition_method', 'none', '--attrition_min_agents', '2', '--attrition_random_probability', '0.0', '--beta', '0.1', '--broadcast_uncertainty_threshold', '0.5', '--clip_param', '0.2', '--communication_model', 'bernoulli', '--communication_probability', '0.005', '--critic_hidden_size', '64', '--critic_lr', '0.0005', '--cuda', '--cuda_deterministic', '--cuda_idx', '2', '--cuda_idx_predictor', '2', '--data_chunk_length', '10', '--no-dec_actor', '--diffusion_autoregression_steps', '1', '--diffusion_epoch', '5', '--diffusion_model_type', 'dit1d', '--diffusion_steps', '20', '--no-encode_state', '--entropy_coef', '0.01', '--env_class', 'patrolling_zoo.patrolling_

/home/anthony/data/dev/qmas_task_allocation/onpolicy/scripts/train/train_pettingzoo.py:158: UserWarning: Unknown arguments: ['--episode_max', '75', '--no-share_reward', '--no-torch_compile', '--no-use_gnn_critic', '--no-use_gnn_mlp_policy']
  warnings.warn(f"Unknown arguments: {unknown_args}")


Pettingzoo environment patrolling_zoo.patrolling_zoo_v0.parallel_env has additional arguments.u are choosing to use rmappo, we set use_recurrent_policy to be True
Pettingzoo environment patrolling_zoo.patrolling_zoo_v0.parallel_env has additional arguments.u are choosing to use rmappo, we set use_recurrent_policy to be True

/home/anthony/data/dev/qmas_task_allocation/onpolicy/scripts/train/train_pettingzoo.py:158: UserWarning: Unknown arguments: ['--episode_max', '69', '--no-share_reward', '--no-torch_compile', '--no-use_gnn_critic', '--no-use_gnn_mlp_policy']
  warnings.warn(f"Unknown arguments: {unknown_args}")




Pettingzoo environment patrolling_zoo.patrolling_zoo_v0.parallel_env has additional arguments.


...

R_Critic: Use CNN: False, Use MLP: TrueR_Critic: Use CNN: False, Use MLP: True

R_Critic: Use CNN: False, Use MLP: True
QmasPolicy: Using diffusion model type dit1d with prediction horizon 8 and transition dimension 130.
Creating ensemble of 3 predictors.
QmasPolicy: Using diffusion model type dit1d with prediction horizon 8 and transition dimension 130.
Creating ensemble of 3 predictors.
QmasPolicy: Using diffusion model type dit1d with prediction horizon 8 and transition dimension 130.
R_Actor: Use GNN: True, Use CNN: False, Use MLP: False, Use RNN: True, Use Encoder: True
State encoder use_cnn: False, input_dim: 130, output_dim: 32
Action output distributions:
  0: Categorical, action_dim: 1, available_action_dim: 2
  1: Categorical, action_dim: 1, available_action_dim: 10
R_Critic: Use CNN: False, Use MLP: True
QmasPolicy: Using diffusion model type dit1d with prediction horizon 8 and transition dimension 130.
QmasPolicy: Using diffusion model type dit1d with prediction horizon 8 a

/home/anthony/data/dev/qmas_task_allocation/onpolicy/scripts/train/train_pettingzoo.py:158: UserWarning: Unknown arguments: ['--episode_max', '75', '--no-share_reward', '--no-torch_compile', '--no-use_gnn_critic', '--no-use_gnn_mlp_policy']
  warnings.warn(f"Unknown arguments: {unknown_args}")


u are choosing to use rmappo, we set use_recurrent_policy to be True
Pettingzoo arguments validated: base
Pettingzoo environment patrolling_zoo.patrolling_zoo_v0.parallel_env has additional arguments validation function.
Pettingzoo arguments validated: patrolling_zoo.patrolling_zoo_v0.parallel_env
PettingzooEnv: Attempting to pass unpacked argparse namespace to environment.
PettingzooEnv: Passing the following arguments to the environment: {'max_cycles': 200, 'num_agents': 4, 'alpha': 1.0, 'beta': 0.1, 'reward_comms_penalty_weight': 0.1, 'graph_file': '/home/anthony/dev/qmas_task_allocation/patrolling_zoo/graphs/cumberland.graph', 'graph_random': True, 'graph_random_nodes': 40, 'reward_method_terminal': 'none', 'reward_interval': 1, 'agent_speed': 40.0, 'action_method': 'neighbors_with_comm_boolean', 'observe_method': 'pyg', 'observe_method_global': 'coordinates', 'observe_bitmap_size': 50, 'observation_radius': 200.0, 'observation_radius_random_min': 0.0, 'observation_radius_random_ma

/home/anthony/miniconda3/envs/qmas/lib/python3.10/site-packages/torchrl/__init__.py:48: UserWarning: failed to set start method to spawn, and current start method for mp is fork.
  warn(


R_Actor: Use GNN: True, Use CNN: False, Use MLP: False, Use RNN: True, Use Encoder: True
State encoder use_cnn: False, input_dim: 260, output_dim: 32
Action output distributions:
  0: Categorical, action_dim: 1, available_action_dim: 2
  1: Categorical, action_dim: 1, available_action_dim: 10
R_Critic: Use CNN: False, Use MLP: True
Creating ensemble of 3 predictors.
QmasPolicy: Using diffusion model type dit1d with prediction horizon 8 and transition dimension 130.
QmasPolicy: Using diffusion model type dit1d with prediction horizon 8 and transition dimension 130.
QmasPolicy: Using diffusion model type dit1d with prediction horizon 8 and transition dimension 130.
Restored models from /data/group/mas/qmas/results/patrolling_zoo_v0/rmappo/patrolling-randomOR-noSkip-comms-pred-uqstd-draft1/wandb/run-20260429_003704-47915ufa/files/episode_55
Initialized QmasAlgorithm with 3 predictors. Threaded training: True
Buffer initialized with episode length 200
Total Pareto evaluation time: 1707.23 

# Analysis

In [ ]:
import zarr
output = zarr.open(output_file, mode='r')
print(output.tree())
print(f"Output file: {os.path.abspath(output_file)}")
print(f"To load results, set in first cell: output_file = '{os.path.abspath(output_file)}'")

In [ ]:
%reload_ext autoreload
%autoreload 2
from plots import *
import numpy as np

x_key = 'observation_radius'

# Data for plotting
plots = {
    'prediction_error_mean': {
        'title': 'Prediction Error',
        'ylabel': 'Prediction Error (units distance)',
        'xlabel': 'Observation Radius',
        'x': {series: get_x_radii(series, output, x_key) for series in output},
        # Average over all episodes and all timesteps.
        'y': {
            series: [np.mean(output[series][exp]['prediction_error_mean']) for exp in sorted_experiments(series, output, x_key)]
                for series in output if not next(iter(output[series].values())).attrs['prediction_disable']
        },
        'y_std': {
            series: [np.std(np.mean(output[series][exp]['prediction_error_mean'], axis=tuple(range(1, output[series][exp]['prediction_error_mean'].ndim)))) for exp in sorted_experiments(series, output, x_key)]
                for series in output if not next(iter(output[series].values())).attrs['prediction_disable']
        }
    },
    'prediction_trajectory_error_mean': {
        'title': 'Prediction Trajectory Error',
        'ylabel': 'Prediction Trajectory Error (units distance)',
        'xlabel': 'Observation Radius',
        'x': {series: get_x_radii(series, output, x_key) for series in output},
        # Average over all episodes and all timesteps.
        'y': {
            series: [np.mean(output[series][exp]['prediction_trajectory_error_mean']) for exp in sorted_experiments(series, output, x_key)]
                for series in output if not next(iter(output[series].values())).attrs['prediction_disable']
        }
    },
    'avg_idleness': {
        'title': 'Time-Averaged Avg. Idleness',
        'ylabel': 'Avg. Node Idleness (s)',
        'xlabel': 'Observation Radius',
        'x': {series: get_x_radii(series, output, x_key) for series in output},
        # Average over all episodes and all timesteps.
        'y': {
            series: [np.mean(output[series][exp]['avg_idleness']) for exp in sorted_experiments(series, output, x_key)]
                for series in output
        },
        'y_std': {
            series: [np.std(np.mean(output[series][exp]['avg_idleness'], axis=tuple(range(1, output[series][exp]['avg_idleness'].ndim)))) for exp in sorted_experiments(series, output, x_key)]
                for series in output
        }
    },
    'avg_idleness_final': {
        'title': 'Final Avg. Idleness',
        'ylabel': 'Avg. Node Idleness (s)',
        'xlabel': 'Observation Radius',
        'x': {series: get_x_radii(series, output, x_key) for series in output},
        'y': {
            series: [np.mean(output[series][exp]['avg_idleness'][:, -1]) for exp in sorted_experiments(series, output, x_key)]
                for series in output
        },
        'y_std': {
            series: [np.std(np.mean(output[series][exp]['avg_idleness'][:, -1], axis=tuple(range(1, output[series][exp]['avg_idleness'][:, -1].ndim)))) for exp in sorted_experiments(series, output, x_key)]
                for series in output
        }
    },
    'avg_idleness_over_time': {
        'title': 'Avg. Idleness vs. Time',
        'ylabel': 'Avg. Node Idleness (s)',
        'xlabel': 'Time Steps',
        # One trace per experiment: average over episodes, keep time dimension.
        'y': {
            series: {
                exp: np.mean(output[series][exp]['avg_idleness'], axis=0)
                for exp in sorted_experiments(series, output, x_key)
            }
            for series in output
        },
        # 'y_std': {
        #     series: {
        #         exp: np.std(output[series][exp]['avg_idleness'], axis=0)
        #         for exp in sorted_experiments(series, output, x_key)
        #     }
        #     for series in output
        # }
    },
    'worst_idleness': {
        'title': 'Time-Averaged Worst Idleness',
        'ylabel': 'Worst Node Idleness (s)',
        'xlabel': 'Observation Radius',
        'x': {series: get_x_radii(series, output, x_key) for series in output},
        # Average over all episodes and all timesteps.
        'y': {
            series: [np.mean(output[series][exp]['worst_idleness']) for exp in sorted_experiments(series, output, x_key)]
                for series in output
        },
        'y_std': {
            series: [np.std(np.mean(output[series][exp]['worst_idleness'], axis=tuple(range(1, output[series][exp]['worst_idleness'].ndim)))) for exp in sorted_experiments(series, output, x_key)]
                for series in output
        }
    },
    'worst_idleness_final': {
        'title': 'Final Worst Idleness',
        'ylabel': 'Worst Node Idleness (s)',
        'xlabel': 'Observation Radius',
        'x': {series: get_x_radii(series, output, x_key) for series in output},
        'y': {
            series: [np.mean(output[series][exp]['worst_idleness'][:, -1]) for exp in sorted_experiments(series, output, x_key)]
                for series in output
        }
    },
    'worst_idleness_over_time': {
        'title': 'Worst Idleness vs. Time',
        'ylabel': 'Worst Node Idleness (s)',
        'xlabel': 'Time Steps',
        # One trace per experiment: average over episodes, keep time dimension.
        'y': {
            series: {
                exp: np.mean(output[series][exp]['worst_idleness'], axis=0)
                for exp in sorted_experiments(series, output, x_key)
            }
            for series in output
        },
        'y_std': {
            series: {
                exp: np.std(output[series][exp]['worst_idleness'], axis=0)
                for exp in sorted_experiments(series, output, x_key)
            }
            for series in output
        }
    },
    'stddev_idleness': {
        'title': 'Time-Averaged Std. Dev. Idleness',
        'ylabel': 'Std. Dev. Node Idleness (s)',
        'xlabel': 'Observation Radius',
        'x': {series: get_x_radii(series, output, x_key) for series in output},
        # Average over all episodes and all timesteps.
        'y': {
            series: [np.mean(output[series][exp]['stddev_idleness']) for exp in sorted_experiments(series, output, x_key)]
                for series in output
        }
    },
    'stddev_idleness_over_time': {
        'title': 'Std. Dev. Idleness vs. Time',
        'ylabel': 'Std. Dev. Node Idleness (s)',
        'xlabel': 'Time Steps',
        # One trace per experiment: average over episodes, keep time dimension.
        'y': {
            series: {
                exp: np.mean(output[series][exp]['stddev_idleness'], axis=0)
                for exp in sorted_experiments(series, output, x_key)
            }
            for series in output
        },
        'y_std': {
            series: {
                exp: np.std(output[series][exp]['stddev_idleness'], axis=0)
                for exp in sorted_experiments(series, output, x_key)
            }
            for series in output
        }
    },
    'communication_requests': {
        'title': 'Communication Requests vs. Observation Radius',
        'ylabel': 'Communication Requests',
        'xlabel': 'Observation Radius',
        'x': {series: get_x_radii(series, output, x_key) for series in output},
        'y': {
            series: [np.mean(output[series][exp]['communication']['requests_made']) for exp in sorted_experiments(series, output, x_key)]
                for series in output
        }
    },
    'communication_requests_time': {
        'title': 'Cumulative Communication Requests vs. Time',
        'ylabel': 'Communication Requests',
        'xlabel': 'Time Steps',
        'y': {
            series: {
                exp: np.cumsum(np.mean(output[series][exp]['communication']['requests_made'], axis=0))
                for exp in sorted_experiments(series, output, x_key)
            }
            for series in output
        }
    },
    'prediction_uncertainty_mean': {
        'title': 'Prediction Uncertainty',
        'ylabel': 'Prediction Uncertainty (units distance)',
        'xlabel': 'Observation Radius',
        'x': {series: get_x_radii(series, output, x_key) for series in output},
        # Average over all episodes and all timesteps.
        'y': {
            series: [np.mean(output[series][exp]['prediction_uncertainty_mean']) for exp in sorted_experiments(series, output, x_key)]
                for series in output if not next(iter(output[series].values())).attrs['prediction_disable']
        }
    },
    'prediction_uncertainty_gap_mean': {
        'title': 'Prediction Uncertainty Gap',
        'ylabel': 'Prediction Uncertainty Gap (units distance)',
        'xlabel': 'Observation Radius',
        'x': {series: get_x_radii(series, output, x_key) for series in output},
        # Average over all episodes and all timesteps.
        'y': {
            series: [np.mean(output[series][exp]['prediction_uncertainty_gap_mean']) for exp in sorted_experiments(series, output, x_key)]
                for series in output if not next(iter(output[series].values())).attrs['prediction_disable']
        }
    },
}

# Miscelaneous statistics.
timesteps = [np.max([np.max(output[series][experiment]['deltaSteps'].shape[1]) for experiment in output[series]]) for series in output]

plot(plots, timesteps=timesteps)


## Pareto Analysis

For each observation radius, plots **final avg. idleness vs. total communication requests**.

- Baseline curves (MAGEC, Prediction): one point per `comms_probability` value, connected as a Pareto frontier.
- Learned comms runs: single star marker at their natural operating point.

A learned-comms point that falls **below and to the left** of the baseline curve achieves better idleness at lower communication cost.

In [4]:
import zarr
import plotly.graph_objects as go
import plotly.express as px
import numpy as np

pareto_output = zarr.open(pareto_output_file, mode='r')
print(pareto_output.tree())

series_colors = {s: c for s, c in zip(pareto_output, px.colors.qualitative.Plotly)}


def get_requests(data):
    """Total communication requests per episode (mean over episodes)."""
    r = np.array(data['communication']['requests_made'][...])
    # Sum over all dims except episodes, then mean over episodes.
    return float(np.mean(r.reshape(r.shape[0], -1).sum(axis=1)))


def get_idleness_final(data):
    """Final avg. idleness (last timestep), mean over episodes."""
    return float(np.mean(data['avg_idleness'][:, -1]))


for obs_radius in pareto_obs_radii:
    fig = go.Figure()

    for series in pareto_output:
        color = series_colors.get(series, '#888888')
        all_exps = list(pareto_output[series].keys())

        # Swept experiments: encoded as obs_radius_X_comms_Y
        swept_exps = sorted(
            [e for e in all_exps if f"obs_radius_{obs_radius:.1f}_comms_" in e],
            key=lambda e: float(e.split("_comms_")[1])
        )
        # Fixed (learned comms) experiments: encoded as obs_radius_X only
        fixed_exps = [e for e in all_exps if e == f"obs_radius_{obs_radius:.1f}"]

        if swept_exps:
            comms_ps = [float(e.split("_comms_")[1]) for e in swept_exps]
            xs = [get_requests(pareto_output[series][e]) for e in swept_exps]
            ys = [get_idleness_final(pareto_output[series][e]) for e in swept_exps]
            fig.add_trace(go.Scatter(
                x=xs,
                y=ys,
                mode='lines+markers',
                name=series,
                line=dict(color=color),
                marker=dict(color=color, size=8),
                legendgroup=series,
                hovertemplate=[
                    f"{series}<br>comms_p={cp:.3f}<br>requests=%{{x:.1f}}<br>idleness=%{{y:.4f}}<extra></extra>"
                    for cp in comms_ps
                ],
            ))

        if fixed_exps:
            xs = [get_requests(pareto_output[series][e]) for e in fixed_exps]
            ys = [get_idleness_final(pareto_output[series][e]) for e in fixed_exps]
            fig.add_trace(go.Scatter(
                x=xs,
                y=ys,
                mode='markers',
                name=series,
                marker=dict(color=color, size=14, symbol='star'),
                legendgroup=series,
                # Only add a legend entry here if no swept trace was already added for this series.
                showlegend=len(swept_exps) == 0,
                hovertemplate=f"{series}<br>requests=%{{x:.1f}}<br>idleness=%{{y:.4f}}<extra></extra>",
            ))

    fig.update_layout(
        title=f"Pareto: Final Avg. Idleness vs. Communication Requests (obs_radius={obs_radius:.1f})",
        xaxis_title="Total Communication Requests (per episode)",
        yaxis_title="Final Avg. Idleness (s)",
        height=600,
        hovermode='closest',
        autosize=True,
        margin=dict(l=60, r=20, t=60, b=120),
        legend=dict(orientation='h', yanchor='top', y=-0.2, xanchor='center', x=0.5),
    )
    fig.show()


/
 ├── Learned Comms. + Prediction
 │   ├── obs_radius_100.0
 │   │   ├── agent_count (3, 200) float32
 │   │   ├── avg_idleness (3, 200) float32
 │   │   ├── communication
 │   │   │   └── requests_made (3, 200) float32
 │   │   ├── deltaSteps (3, 200, 4, 1) float32
 │   │   ├── node_visits
 │   │   │   ├── node_0 (3, 200) float32
 │   │   │   ├── node_1 (3, 200) float32
 │   │   │   ├── node_10 (3, 200) float32
 │   │   │   ├── node_11 (3, 200) float32
 │   │   │   ├── node_12 (3, 200) float32
 │   │   │   ├── node_13 (3, 200) float32
 │   │   │   ├── node_14 (3, 200) float32
 │   │   │   ├── node_15 (3, 200) float32
 │   │   │   ├── node_16 (3, 200) float32
 │   │   │   ├── node_17 (3, 200) float32
 │   │   │   ├── node_18 (3, 200) float32
 │   │   │   ├── node_19 (3, 200) float32
 │   │   │   ├── node_2 (3, 200) float32
 │   │   │   ├── node_20 (3, 200) float32
 │   │   │   ├── node_21 (3, 200) float32
 │   │   │   ├── node_22 (3, 200) float32
 │   │   │   ├── node_23 (3, 200) floa